# Lecture 08 作业：Kubernetes 原生 Spark Connect

学号：`20234080203`　姓名：叶平

本 Notebook 按 `HW-lecture08.pdf` 要求，使用 **Kubernetes 原生 Spark Connect** 完成作业：Spark Connect Server 运行在 Kubernetes 中，Executor 由 Kubernetes 调度为 Pod。不使用 Spark Standalone 的 Master/Worker。

**作业对照**

1. 第一个代码框打印实际集群信息（context、kube-system UID、节点名称/状态/Internal IP、Spark Connect 资源），证明不是本地伪造运行。
2. 第二个代码框通过 Spark Connect 提交不少于 `1_000_000` 个采样点的 π 计算。
3. 不提交 kubeconfig、SSH 私钥、密码或访问令牌。
4. 提交文件：`20234080203-叶平-lecture08.ipynb`



## 1. 连接 Kubernetes 并打印集群信息

优先通过 WSL 中的 Kind 集群执行作业模板；若本机 `kubectl` 已指向云集群（如腾讯云 kubeadm），则走云 Kubernetes 模板。



In [1]:
import hashlib
import json
import subprocess
import os
import shutil
_BIN = r'C:\data_wave\bin'
if os.path.isdir(_BIN):
    os.environ['PATH'] = _BIN + os.pathsep + os.environ.get('PATH', '')
os.environ.setdefault(
    'KUBECONFIG', os.path.expandvars(r'%USERPROFILE%\.kube\config')
)
from pyspark.sql import SparkSession, functions as F


def _run(cmd):
    try:
        return subprocess.run(
            cmd, text=True, capture_output=True, encoding="utf-8", errors="replace"
        )
    except FileNotFoundError as exc:
        class _Missing:
            returncode = 127
            stdout = ""
            stderr = str(exc)
        return _Missing()


def kubectl(*args):
    # Windows 默认 kubeconfig 可能指向其他集群；优先通过 WSL 使用 Kind 的 kubectl。
    wsl = _run(["wsl.exe", "--", "kubectl", *args])
    if wsl.returncode == 0:
        kubectl.backend = "wsl-kind"
        return wsl.stdout
    kubectl_exe = shutil.which("kubectl") or os.path.join(_BIN, "kubectl.exe")
    native = _run([kubectl_exe, *args])
    if native.returncode == 0:
        kubectl.backend = "native"
        return native.stdout
    err = (wsl.stderr or "") + "\n" + (native.stderr or "")
    raise RuntimeError(
        "无法调用 kubectl。请先在 WSL 中部署 Kind + Spark Connect，"
        "或配置云 Kubernetes 的 kubeconfig。\n" + err.strip()
    )


context = kubectl("config", "current-context").strip()
cluster_uid = kubectl(
    "get", "namespace", "kube-system", "-o", "jsonpath={.metadata.uid}"
).strip()
backend = getattr(kubectl, "backend", "unknown")
print(f"kubectl backend: {backend}")
print(f"Current Kubernetes context: {context}")
print(f"Cluster UID (kube-system namespace UID): {cluster_uid}")

nodes = json.loads(kubectl("get", "nodes", "-o", "json"))["items"]
node_keys = []
print("\nNode inventory:")
for node in nodes:
    addresses = {a["type"]: a["address"] for a in node["status"].get("addresses", [])}
    name = node["metadata"]["name"]
    uid = node["metadata"]["uid"]
    status = next(
        (c["type"] for c in node["status"].get("conditions", []) if c.get("status") == "True"),
        "Unknown",
    )
    ready = next(
        (c["status"] for c in node["status"].get("conditions", []) if c.get("type") == "Ready"),
        "Unknown",
    )
    runtime = node["status"].get("nodeInfo", {}).get("containerRuntimeVersion", "unknown")
    print(
        f"- {name}: status=Ready={ready}/{status}, "
        f"internal_ip={addresses.get('InternalIP')}, uid={uid[:8]}..., runtime={runtime}"
    )
    node_keys.append(f"{uid}:{addresses.get('InternalIP')}")

fingerprint = hashlib.sha256("|".join(sorted(node_keys)).encode()).hexdigest()[:16]
print(f"Node-inventory fingerprint: {fingerprint}")

print("\nCluster information:\n" + kubectl("cluster-info"))
print("Nodes (wide):\n" + kubectl("get", "nodes", "-o", "wide"))
print("Namespaces:\n" + kubectl("get", "namespaces"))

connect_pods = json.loads(
    kubectl("-n", "spark", "get", "pods", "-l", "app=spark-connect", "-o", "json")
)["items"]
print("\nSpark Connect Pod placement:")
if not connect_pods:
    print("- (namespace spark 中没有 label=app=spark-connect 的 Pod)")
for pod in connect_pods:
    print(
        f"- {pod['metadata']['name']}: node={pod['spec'].get('nodeName')}, "
        f"pod_ip={pod['status'].get('podIP')}, phase={pod['status'].get('phase')}"
    )
print(
    "\nSpark Connect resources:\n"
    + kubectl("-n", "spark", "get", "deployment,service,pods")
)


kubectl backend: native
Current Kubernetes context: default
Cluster UID (kube-system namespace UID): e0178398-ce47-46bb-820d-81d49df6769b



Node inventory:
- vm-0-8-ubuntu: status=Ready=True/Ready, internal_ip=2402:4e00:c013:8a00:585a:f530:ea16:0, uid=e5fdb579..., runtime=containerd://2.3.4-k3s1.36
Node-inventory fingerprint: 21e143307ae0324d



Cluster information:
Kubernetes control plane is running at https://127.0.0.1:6443
CoreDNS is running at https://127.0.0.1:6443/api/v1/namespaces/kube-system/services/kube-dns:dns/proxy
Metrics-server is running at https://127.0.0.1:6443/api/v1/namespaces/kube-system/services/https:metrics-server:https/proxy

To further debug and diagnose cluster problems, use 'kubectl cluster-info dump'.



Nodes (wide):
NAME            STATUS   ROLES           AGE   VERSION        INTERNAL-IP   EXTERNAL-IP   OS-IMAGE           KERNEL-VERSION               CONTAINER-RUNTIME
vm-0-8-ubuntu   Ready    control-plane   31m   v1.36.5+k3s1   10.1.0.8      <none>        Ubuntu 22.04 LTS   5.15.0-160-generic (amd64)   containerd://2.3.4-k3s1.36



Namespaces:
NAME              STATUS   AGE
default           Active   31m
kube-node-lease   Active   31m
kube-public       Active   31m
kube-system       Active   31m
spark             Active   24m




Spark Connect Pod placement:
- spark-connect-5dd789bb96-bmwd7: node=vm-0-8-ubuntu, pod_ip=10.42.0.15, phase=Running



Spark Connect resources:
NAME                            READY   UP-TO-DATE   AVAILABLE   AGE
deployment.apps/spark-connect   1/1     1            1           24m

NAME                    TYPE        CLUSTER-IP      EXTERNAL-IP   PORT(S)                       AGE
service/spark-connect   ClusterIP   10.43.138.100   <none>        15002/TCP,7078/TCP,7079/TCP   24m

NAME                                 READY   STATUS    RESTARTS   AGE
pod/spark-connect-5dd789bb96-bmwd7   1/1     Running   0          13m
pod/spark-exec-exec-3                1/1     Running   0          12m



## 2. 通过 Spark Connect 计算 π

客户端连接到集群中的 Spark Connect Server。作业示例端口为 `15002`；若 Service 端口不同，可修改 `SPARK_CONNECT_URL`。采样点数不少于 `1_000_000`。



In [2]:
import os

n = 1_000_000
connect_url = os.getenv("SPARK_CONNECT_URL", "sc://localhost:15002")
print(f"Spark Connect URL: {connect_url}")
print(f"Sample points: {n:,}")

spark = SparkSession.builder.remote(connect_url).getOrCreate()
x = (F.col("id") + F.lit(0.5)) / F.lit(float(n))
pi = (
    spark.range(n)
    .select((F.lit(4.0) * F.sqrt(F.lit(1.0) - x * x) / F.lit(float(n))).alias("area"))
    .agg(F.sum("area").alias("pi"))
    .first()["pi"]
)
print(f"Pi is roughly {pi}")
spark.stop()


Spark Connect URL: sc://localhost:15002
Sample points: 1,000,000


Pi is roughly 3.1415926539341976


## 提交说明

- 文件名：`20234080203-叶平-lecture08.ipynb`
- 保留本 Notebook 中两个代码框的运行输出
- 不要提交 `admin.conf`、kubeconfig、SSH 私钥、密码或访问令牌

